# Đọc CV (PDF, tiếng Anh) → Structure-based chunking + Semantic chunking

**Ý tưởng chung**

1. **Đọc PDF** bằng PyMuPDF, giữ lại cỡ chữ / chữ đậm của từng dòng.
2. **Structure-based chunking** (theo cấu trúc): nhận diện các mục *Summary, Experience, Education, Skills, Projects...* rồi cắt tiếp thành từng *entry* (mỗi công việc / mỗi trường / mỗi dự án là một chunk).
3. **Semantic chunking** (theo ngữ nghĩa): với chunk còn quá dài, dùng embedding để cắt tại chỗ ý nghĩa đổi chủ đề.
4. **Hybrid**: cấu trúc trước, ngữ nghĩa sau. Mỗi chunk con vẫn giữ tiêu đề (ví dụ tên công ty + chức danh) để không mất ngữ cảnh.

> Chạy lần lượt từng ô từ trên xuống. Có sẵn một CV mẫu để thử; muốn dùng CV thật thì làm ở ô số 3.

## 1. Cài thư viện

In [ ]:
!pip install -q pymupdf sentence-transformers reportlab pandas numpy


## 2. Khai báo cấu trúc dữ liệu & hàm đọc PDF

In [ ]:
import re, json, statistics
from dataclasses import dataclass, field, asdict
from typing import Callable, List, Dict, Optional
import numpy as np
try:
    import pymupdf as fitz      # PyMuPDF (tên mới)
except ImportError:
    import fitz


@dataclass
class Line:
    """Một dòng chữ trong PDF kèm thông tin định dạng."""
    text: str
    size: float      # cỡ chữ
    bold: bool       # có in đậm không
    page: int


def read_pdf_lines(path: str) -> List[Line]:
    """Đọc PDF -> danh sách dòng (đúng thứ tự đọc), kèm cỡ chữ và chữ đậm."""
    lines: List[Line] = []
    with fitz.open(path) as doc:
        for page_no, page in enumerate(doc, start=1):
            data = page.get_text("dict", sort=True)          # sort=True: sắp theo thứ tự đọc
            for block in data["blocks"]:
                if block.get("type") != 0:                   # 0 = khối chữ (bỏ qua ảnh)
                    continue
                for ln in block["lines"]:
                    spans = [s for s in ln["spans"] if s["text"].strip()]
                    if not spans:
                        continue
                    text = re.sub(r"\s+", " ", " ".join(s["text"].strip() for s in spans)).strip()
                    size = max(s["size"] for s in spans)
                    bold = any((s["flags"] & 16) or "bold" in s["font"].lower() for s in spans)
                    lines.append(Line(text, round(size, 1), bool(bold), page_no))
    return lines


## 3. Tạo CV mẫu (hoặc upload CV của bạn)

Chạy ô này để có file `sample_cv.pdf`. Muốn dùng CV thật: bỏ dấu `#` ở 3 dòng cuối để upload, rồi đặt `PDF_PATH` là tên file.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer

def make_sample_cv(path="sample_cv.pdf"):
    ss = getSampleStyleSheet()
    name = ParagraphStyle("name", parent=ss["Title"], fontSize=20, alignment=0)
    h = ParagraphStyle("h", parent=ss["Heading2"], fontSize=12, textTransform="uppercase", spaceBefore=10)
    body = ParagraphStyle("b", parent=ss["BodyText"], fontSize=10, leading=13)
    bold = ParagraphStyle("t", parent=body, fontName="Helvetica-Bold")
    B = lambda s: Paragraph("&bull; " + s, ParagraphStyle("bl", parent=body, leftIndent=12, firstLineIndent=-8))
    P = lambda s, st=body: Paragraph(s, st)
    story = [
        P("Alex Nguyen", name),
        P("alex.nguyen@email.com | +84 912 345 678 | Can Tho, Vietnam | linkedin.com/in/alexnguyen"),
        P("Summary", h),
        P("Data scientist with 5 years of experience building NLP and recommendation systems. "
          "Passionate about turning messy text data into reliable products. "
          "Enjoys mentoring junior engineers and writing clear technical documentation."),
        P("Work Experience", h),
        P("Senior Data Scientist - TechNova Corp", bold),
        P("Jan 2022 - Present | Ho Chi Minh City"),
        B("Led a team of four engineers to build a resume-screening engine that reduced manual review time by 60%."),
        B("Fine-tuned transformer models for named entity recognition on 200k documents, improving F1 from 0.78 to 0.89."),
        B("Designed a vector search service with FAISS and FastAPI that serves 50 queries per second."),
        B("Organised weekly knowledge-sharing sessions and onboarded six new hires."),
        B("Collaborated with product managers to define success metrics and A/B test new ranking features."),
        P("Data Analyst - FinSmart Ltd", bold),
        P("Jun 2019 - Dec 2021 | Hanoi"),
        B("Built dashboards in Tableau and Power BI that tracked loan performance for 12 regional branches."),
        B("Automated monthly reporting with Python and SQL, saving about 30 hours per month."),
        P("Education", h),
        P("M.Sc. in Computer Science - Can Tho University", bold),
        P("2017 - 2019"),
        B("Thesis: Sentence simplification for Vietnamese using sequence-to-sequence models."),
        P("B.Sc. in Information Technology - Can Tho University", bold),
        P("2013 - 2017"),
        P("Skills", h),
        P("Programming: Python, SQL, R, JavaScript"),
        P("Machine Learning: PyTorch, scikit-learn, Hugging Face Transformers, XGBoost"),
        P("Tools: Docker, Git, Airflow, Tableau"),
        P("Projects", h),
        P("Open-source CV Parser", bold),
        P("2023"),
        B("Created a Python library that extracts sections from PDF resumes and exports them as JSON."),
        P("Certifications", h),
        P("AWS Certified Machine Learning - Specialty (2023)"),
        P("Languages", h),
        P("English (fluent), Vietnamese (native)"),
    ]
    SimpleDocTemplate(path, pagesize=A4, leftMargin=50, rightMargin=50, topMargin=40, bottomMargin=40).build(story)
    return path

PDF_PATH = make_sample_cv()

# --- Dùng CV thật của bạn: bỏ dấu # ---
# from google.colab import files
# up = files.upload()
# PDF_PATH = list(up.keys())[0]
print("PDF:", PDF_PATH)


## 4. Bước 1 – Nhận diện các mục (Section) của CV

Có hai cách nhận diện tiêu đề mục:
- **Theo từ khoá**: dòng ngắn như `Work Experience`, `Education`, `Skills`... (đáng tin nhất).
- **Theo kiểu chữ** (dự phòng): dòng ngắn, VIẾT HOA hoặc chữ to hơn thân bài — chỉ dùng sau khi đã gặp mục đầu tiên.

Phần trước mục đầu tiên (tên, email, điện thoại...) được gom vào mục `HEADER`.

In [ ]:
SECTION_PATTERNS: Dict[str, str] = {
    "SUMMARY":        r"(professional |career )?(summary|profile|objective)|about me",
    "EXPERIENCE":     r"((work|professional|relevant) )?experience|employment( history)?|work history|internships?",
    "EDUCATION":      r"education( and training)?|academic background",
    "SKILLS":         r"((technical|key|core) )?skills|competencies|technologies",
    "PROJECTS":       r"((personal|selected|academic) )?projects",
    "CERTIFICATIONS": r"certifications?|licenses?( and certifications?)?|courses",
    "AWARDS":         r"awards?|honou?rs|achievements",
    "PUBLICATIONS":   r"publications?",
    "LANGUAGES":      r"languages?",
    "INTERESTS":      r"interests|hobbies",
    "REFERENCES":     r"references?",
}
ENTRY_SECTIONS = {"EXPERIENCE", "EDUCATION", "PROJECTS"}   # các mục gồm nhiều "entry" (mỗi entry = 1 chunk)
PROSE_SECTIONS = {"SUMMARY", "OTHER"}                      # các mục là văn xuôi (nối dòng rồi tách câu)


def match_section_keyword(text: str) -> Optional[str]:
    t = re.sub(r"[^a-zA-Z& ]", " ", text).lower()
    t = re.sub(r"\s+", " ", t).strip()
    if not t or len(t.split()) > 5:
        return None
    for name, pat in SECTION_PATTERNS.items():
        if re.fullmatch(pat, t):
            return name
    return None


def looks_like_heading_by_style(line: Line, body_size: float) -> bool:
    words = line.text.split()
    if not (1 <= len(words) <= 4) or any(ch.isdigit() for ch in line.text) or line.text.endswith((".", ",")):
        return False
    is_caps = line.text.isupper() and len(line.text) > 3
    is_bigger = line.size >= body_size + 1.0
    return is_caps or is_bigger


@dataclass
class Section:
    name: str                 # SUMMARY / EXPERIENCE / ...
    title: str                # tiêu đề gốc trong CV
    lines: List[Line] = field(default_factory=list)


def split_sections(lines: List[Line]) -> List[Section]:
    body_size = statistics.median(l.size for l in lines)
    sections = [Section("HEADER", "HEADER")]
    seen_keyword = False
    for ln in lines:
        name = match_section_keyword(ln.text)
        if name:
            sections.append(Section(name, ln.text.strip()))
            seen_keyword = True
        elif seen_keyword and looks_like_heading_by_style(ln, body_size):
            sections.append(Section("OTHER", ln.text.strip()))
        else:
            sections[-1].lines.append(ln)
    return [s for s in sections if s.lines]


## 5. Bước 2 – Structure-based chunking

Quy tắc:
- `HEADER`, `SUMMARY`, `SKILLS`, `CERTIFICATIONS`, ... → **mỗi mục là 1 chunk**.
- `EXPERIENCE`, `EDUCATION`, `PROJECTS` → **mỗi entry là 1 chunk**. Một entry mới bắt đầu khi gặp dòng tiêu đề (không phải gạch đầu dòng) đứng sau các gạch đầu dòng.
- Dòng bị xuống hàng giữa chừng của một gạch đầu dòng sẽ được nối lại.
- Mỗi chunk có `header` (ví dụ *Senior Data Scientist - TechNova Corp, Jan 2022 - Present*) và `units` (các gạch đầu dòng / câu).

In [ ]:
BULLET_RE = re.compile(r"^\s*([•●▪■◦·‣▸►*]|-|–|\d+[.)])\s+")
MONTH = r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\.?"
DATE_RE = re.compile(rf"(?:{MONTH}\s+)?(?:19|20)\d{{2}}\s*(?:-|–|—|to)\s*(?:(?:{MONTH}\s+)?(?:19|20)\d{{2}}|present|current|now)",
                     re.IGNORECASE)
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")


def is_bullet(text: str) -> bool:
    return bool(BULLET_RE.match(text))


def strip_bullet(text: str) -> str:
    return BULLET_RE.sub("", text).strip()


@dataclass
class Chunk:
    section: str                      # EXPERIENCE, EDUCATION, ...
    header: str                       # ngữ cảnh đi kèm (tiêu đề mục/entry)
    units: List[str]                  # các đơn vị nội dung (bullet/câu)
    source: str = "structure"         # structure | structure+semantic
    page: int = 1
    index: int = 0

    @property
    def body(self) -> str:
        return "\n".join(self.units)

    @property
    def text(self) -> str:
        return f"{self.header}\n{self.body}".strip()


def _is_continuation(prev: str, cur: str) -> bool:
    """Dòng `cur` có phải là phần nối tiếp (bị xuống hàng) của `prev` không?"""
    if is_bullet(cur) or DATE_RE.search(cur):
        return False
    if cur[:1].islower():
        return True
    return len(prev) >= 70 and prev[-1] not in ".;:!?"      # dòng trước dài sát lề & chưa kết thúc câu


def split_entries(lines: List[Line]) -> List[Dict]:
    """Chia dòng của mục EXPERIENCE/EDUCATION/PROJECTS thành các entry {header, units, page}."""
    entries: List[Dict] = []
    cur: Optional[Dict] = None
    for ln in lines:
        t = ln.text
        if cur is None:
            cur = {"header": [t], "units": [], "page": ln.page}
            continue
        if cur["units"]:                                   # entry đã có bullet
            if is_bullet(t):
                cur["units"].append(strip_bullet(t))
            elif _is_continuation(cur["units"][-1], t):
                cur["units"][-1] += " " + t
            else:                                          # dòng tiêu đề mới -> entry mới
                entries.append(cur)
                cur = {"header": [t], "units": [], "page": ln.page}
        else:                                              # đang ở phần tiêu đề của entry
            if is_bullet(t):
                cur["units"].append(strip_bullet(t))
            elif DATE_RE.search(cur["header"][-1]) and DATE_RE.search(t) is None and not t[:1].islower() \
                    and len(cur["header"]) >= 2 and not is_bullet(t):
                entries.append(cur)                        # entry trước không có bullet (vd. bằng cấp) -> tách
                cur = {"header": [t], "units": [], "page": ln.page}
            else:
                cur["header"].append(t)
    if cur:
        entries.append(cur)
    return entries


def structure_chunk(lines: List[Line]) -> List[Chunk]:
    """CV -> danh sách Chunk theo cấu trúc (chưa dùng ngữ nghĩa)."""
    chunks: List[Chunk] = []
    for sec in split_sections(lines):
        page = sec.lines[0].page
        if sec.name in ENTRY_SECTIONS:
            for e in split_entries(sec.lines):
                header = f"[{sec.name}] " + " | ".join(e["header"])
                chunks.append(Chunk(sec.name, header, e["units"], page=e["page"]))
        else:
            if sec.name in PROSE_SECTIONS:                 # văn xuôi: nối các dòng bị xuống hàng, rồi tách câu
                paras: List[str] = []
                for ln in sec.lines:
                    if not paras or is_bullet(ln.text):
                        paras.append(strip_bullet(ln.text))
                    else:
                        paras[-1] += " " + ln.text
                units = [s.strip() for p in paras for s in SENT_SPLIT_RE.split(p) if s.strip()]
            else:                                          # dạng danh sách (Skills...): mỗi dòng là một unit
                units = [strip_bullet(ln.text) for ln in sec.lines]
            title = "" if sec.title.upper() == sec.name else f" {sec.title}"
            chunks.append(Chunk(sec.name, f"[{sec.name}]{title}", units, page=page))
    for i, c in enumerate(chunks):
        c.index = i
    return chunks


### Chạy thử Structure-based chunking

In [ ]:
lines = read_pdf_lines(PDF_PATH)
print(f"Đọc được {len(lines)} dòng\n")

structure_chunks = structure_chunk(lines)
for c in structure_chunks:
    print(f"--- #{c.index} | {c.section} | {len(c.text)} ký tự | {len(c.units)} units")
    print(c.text, "\n")


## 6. Bước 3 – Semantic chunking

Với một danh sách các đơn vị (câu/bullet) ta làm như sau:
1. Biến mỗi đơn vị thành **vector embedding** (model `all-MiniLM-L6-v2`, nhẹ và tốt cho tiếng Anh).
2. Tính **khoảng cách ngữ nghĩa** giữa hai đơn vị liền kề: `1 − cosine`.
3. Chỗ nào khoảng cách **lớn bất thường** (trên ngưỡng phân vị `breakpoint_percentile`) → **cắt**.
4. Gộp nhóm quá ngắn (`min_chars`) và tách nhóm quá dài (`max_chars`).

`window=1` nghĩa là mỗi đơn vị được nhúng cùng với đơn vị đứng trước/sau để so sánh mượt hơn.

In [ ]:
class SemanticChunker:
    def __init__(self, embed_fn: Callable[[List[str]], np.ndarray],
                 breakpoint_percentile: float = 70, min_distance: float = 0.15,
                 min_chars: int = 150, max_chars: int = 600, window: int = 1):
        self.embed_fn = embed_fn
        self.p = breakpoint_percentile        # cắt ở các khoảng cách thuộc top (100-p)%
        self.min_distance = min_distance      # nhưng khoảng cách phải lớn hơn mức tối thiểu này
        self.min_chars, self.max_chars, self.window = min_chars, max_chars, window

    # ---- 1. tính khoảng cách giữa các đơn vị liền kề ----
    def distances(self, units: List[str]) -> np.ndarray:
        w = self.window
        ctx = [" ".join(units[max(0, i - w): i + w + 1]) for i in range(len(units))]
        e = np.asarray(self.embed_fn(ctx), dtype=float)
        e = e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-9)
        return 1.0 - np.sum(e[:-1] * e[1:], axis=1)

    # ---- 2. cắt tại điểm đổi chủ đề ----
    def split(self, units: List[str]) -> List[List[str]]:
        if len(units) <= 1:
            return [units]
        d = self.distances(units)
        thr = max(float(np.percentile(d, self.p)), self.min_distance)
        groups, cur = [], [units[0]]
        for i, dist in enumerate(d):
            if dist > thr:
                groups.append(cur)
                cur = []
            cur.append(units[i + 1])
        groups.append(cur)
        return self._enforce_size(groups)

    # ---- 3. đảm bảo kích thước hợp lý ----
    def _size(self, g): return len("\n".join(g))

    def _enforce_size(self, groups: List[List[str]]) -> List[List[str]]:
        merged: List[List[str]] = []
        for g in groups:                                   # gộp nhóm quá ngắn vào nhóm trước
            if merged and self._size(g) < self.min_chars and self._size(merged[-1]) + self._size(g) <= self.max_chars:
                merged[-1] = merged[-1] + g
            else:
                merged.append(g)
        if len(merged) > 1 and self._size(merged[0]) < self.min_chars and \
                self._size(merged[0]) + self._size(merged[1]) <= self.max_chars:
            merged[1] = merged[0] + merged[1]              # nhóm đầu quá ngắn -> gộp xuống nhóm sau
            merged.pop(0)
        out: List[List[str]] = []
        for g in merged:                                   # tách nhóm quá dài theo từng unit
            cur: List[str] = []
            for u in g:
                if cur and self._size(cur + [u]) > self.max_chars:
                    out.append(cur)
                    cur = []
                cur.append(u)
            if cur:
                out.append(cur)
        return out


### Nạp model embedding

In [ ]:
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embed = lambda texts: st_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

semantic = SemanticChunker(embed, breakpoint_percentile=70, min_distance=0.15, min_chars=150, max_chars=600)


## 7. Bước 4 – Hybrid: Structure → Semantic

Chỉ những chunk **dài hơn `max_chars`** mới bị cắt thêm bằng ngữ nghĩa. Các chunk con **giữ nguyên `header`** của entry gốc nên khi đưa vào RAG vẫn biết thuộc công ty/chức danh nào.

In [ ]:
def hybrid_chunk(chunks: List[Chunk], chunker: SemanticChunker, max_chars: int = 600) -> List[Chunk]:
    out: List[Chunk] = []
    for c in chunks:
        if len(c.body) <= max_chars or len(c.units) < 2:
            out.append(c)
            continue
        for group in chunker.split(c.units):
            out.append(Chunk(c.section, c.header, group, source="structure+semantic", page=c.page))
    for i, c in enumerate(out):
        c.index = i
    return out


final_chunks = hybrid_chunk(structure_chunks, semantic, max_chars=400)   # đặt 400 để thấy hiệu ứng trên CV mẫu
print(f"Structure: {len(structure_chunks)} chunks  ->  Hybrid: {len(final_chunks)} chunks\n")
for c in final_chunks:
    print(f"--- #{c.index} | {c.section} | {c.source} | {len(c.text)} ký tự")
    print(c.text, "\n")


### Xem khoảng cách ngữ nghĩa (để hiểu và chỉnh ngưỡng)

In [ ]:
import pandas as pd
exp = max(structure_chunks, key=lambda c: len(c.body))       # chunk dài nhất
d = semantic.distances(exp.units)
print(exp.header, "\n")
pd.DataFrame({"unit": [u[:70] + "..." for u in exp.units[:-1]],
              "khoảng cách tới unit kế tiếp": d.round(3)})


## 8. Xuất kết quả (JSONL cho RAG / vector DB)

In [ ]:
def export_jsonl(chunks: List[Chunk], path: str = "cv_chunks.jsonl"):
    with open(path, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps({"id": c.index, "section": c.section, "page": c.page, "source": c.source,
                                "header": c.header, "text": c.text}, ensure_ascii=False) + "\n")
    return path

path = export_jsonl(final_chunks)
print("Đã lưu:", path)
# from google.colab import files; files.download(path)


## 9. Hàm tiện ích: xử lý nhiều CV một lần

In [ ]:
def process_cv(pdf_path: str, max_chars: int = 600, use_semantic: bool = True) -> List[Chunk]:
    chunks = structure_chunk(read_pdf_lines(pdf_path))
    return hybrid_chunk(chunks, semantic, max_chars) if use_semantic else chunks

# Ví dụ:
# for pdf in ["cv1.pdf", "cv2.pdf"]:
#     export_jsonl(process_cv(pdf), pdf.replace(".pdf", "_chunks.jsonl"))


## Ghi chú & giới hạn

- CV **scan (ảnh)** không có lớp chữ → cần OCR (ví dụ `pytesseract`) trước khi dùng notebook này.
- CV **2 cột** hoặc có bảng phức tạp: PyMuPDF đọc theo thứ tự khối chữ nên đôi khi bị lẫn cột; có thể thử `page.get_text("blocks")` và tự sắp xếp theo toạ độ.
- Muốn thêm mục lạ (vd. *Volunteering*), thêm một dòng vào `SECTION_PATTERNS`.
- Chỉnh chất lượng cắt ngữ nghĩa bằng `breakpoint_percentile` (cao hơn → ít điểm cắt hơn), `min_chars`, `max_chars`.
- Với RAG: lưu `header` làm metadata, và embed `text` (đã gồm header) của từng chunk.